# Retail sales exploration
Run `python -m retail.cli demo` first. This notebook reads the generated SQLite store. The default dataset is synthetic; use real-data outputs only with their provenance.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / 'artifacts').exists() else Path.cwd().parent
conn = sqlite3.connect(ROOT / 'artifacts/sales.sqlite')
daily = pd.read_sql_query('SELECT * FROM daily_sales ORDER BY date, sku', conn, parse_dates=['date'])
daily.head()

## SQL product summary
Which products dominate sales volume? Does a pooled metric hide low-volume products?

In [ ]:
summary = pd.read_sql_query('SELECT sku, SUM(quantity) AS units, AVG(quantity) AS daily_mean FROM daily_sales GROUP BY sku ORDER BY units DESC', conn)
summary.head(10)

In [ ]:
daily.groupby('date').quantity.sum().plot(figsize=(12, 4), title='Observed daily sales across products')
plt.ylabel('Units')
plt.show()

## Weekly pattern and zero-sale days
A weekly pattern motivates the seasonal-naive baseline. Zero observed sales are not proof of zero demand.

In [ ]:
daily['weekday'] = daily.date.dt.day_name()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily.groupby('weekday').quantity.mean().reindex(order).plot.bar(title='Mean daily unit sales by weekday')
plt.show()
print('Zero-sale share:', daily.quantity.eq(0).mean())

## Holdout error by SKU
Inspect large errors and systematic bias before translating predictions into orders.

In [ ]:
errors = pd.read_csv(ROOT / 'reports/metrics_by_sku.csv')
errors[errors.split.eq('holdout')].sort_values('mae', ascending=False).head(10)

## Questions to answer in your portfolio
1. Where does the baseline outperform ML?
2. Which products have persistent negative bias?
3. How does accuracy change across the 28-day horizon?
4. Which data would help distinguish stockouts from weak demand?


In [ ]:
horizons = pd.read_csv(ROOT / 'reports/metrics_by_horizon.csv')
horizons[horizons.split.eq('holdout')].pivot(index='horizon', columns='model', values='mae').plot(title='Holdout MAE by horizon')
plt.show()
conn.close()